# Traffic DRL Orchestration Demo

This notebook demonstrates how to use the centralized configuration and run ID management systems
for training and evaluation orchestration in the traffic DRL project.
You can run this notebook locally or directly on Google Colab.


## 1. Setup Colab Environment

Run this cell only if you are executing this notebook on Google Colab. It clones the project repository, installs headless SUMO binaries, and installs the project as a package along with its dependencies.

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    # Clone the repository and set working directory
    if not os.path.exists('DRL-Traffic-Signal-Control'):
        !git clone https://github.com/TCL03-HCMUT/DRL-Traffic-Signal-Control.git
    %cd DRL-Traffic-Signal-Control
    
    # Install SUMO native binaries and dependencies
    !sudo add-apt-repository ppa:sumo/stable -y
    !sudo apt-get update
    !sudo apt-get install -y sumo sumo-tools sumo-doc
    os.environ['SUMO_HOME'] = '/usr/share/sumo'
    
    # Install colab-specific dependencies and the project package itself
    !pip install -r requirements-colab.txt
    !pip install -e .

## 2. Import Required Modules


In [ ]:
# Import configuration management
from traffic_drl.config import load_env_config, load_train_config, load_eval_config

# Import run ID management
from traffic_drl.run_id import (
    get_default_run_id,
    ensure_run_directories,
)

from traffic_drl.train.train_dqn import run_dqn_pilot
from traffic_drl.evaluation.evaluate_benchmark import evaluate_manifest
from traffic_drl.environment.make_env import create_sumo_env, make_vectorized_environment, build_reward_fn
from traffic_drl.environment.scenario_factory import ScenarioManifest


## 3. Load Configurations


In [ ]:
# Load training configuration
train_config = load_train_config("configs/train/dqn_dummy.yaml")
print("Training config loaded:")
print(f"  Experiment: {train_config.experiment.name}")
print(f"  Seed: {train_config.experiment.seed}")
print(f"  Device: {train_config.experiment.device}")
print()

# Load evaluation configuration
eval_config = load_eval_config("configs/evaluation/phase1_validation.yaml")
print("Evaluation config loaded:")
print(f"  Benchmark: {eval_config.benchmark.name}")
print(f"  Split: {eval_config.benchmark.split}")
print(f"  Deterministic: {eval_config.benchmark.deterministic}")


## 4. Live TensorBoard Monitoring


In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/runs/


## 5. Train the Model

We use `run_dqn_pilot` which self-contains environment creation, wrappers, smoke testing, and checkpoint logic.

In [ ]:
# Generate a run ID for this experiment
run_id = get_default_run_id(prefix="dqn_phase1")
print(f"Generated run ID: {run_id}")

tripinfo_dir, results_dir, checkpoint_dir, logs_dir = ensure_run_directories(run_id)
print(f"Results directory: {results_dir}")

# Train the model using the orchestration function
trained_model = run_dqn_pilot(
    config=train_config,
    checkpoint_dir=checkpoint_dir,
    total_timesteps=train_config.training_control.total_timesteps,
    seed=train_config.experiment.seed
)
print("Model training completed.")

# Save the final model explicitly
from traffic_drl.train.checkpointing import save_checkpoint

# Save final model as a unified checkpoint bundle
final_checkpoint_dir = results_dir / "final_checkpoint"
save_checkpoint(
    trained_model,
    final_checkpoint_dir,
    vec_normalize_env=trained_model.get_vec_normalize_env(),
    save_replay_buffer=True,
    resume_info={"model_class": "DQN", "num_timesteps": 10000}
)
print(f"Final checkpoint saved at {final_checkpoint_dir}")


## 6. Run Evaluation/Benchmark


In [ ]:
from stable_baselines3 import DQN

# Evaluate across validation scenarios
manifest = ScenarioManifest.from_csv(eval_config.manifest_path)

def eval_env_factory(record, seed):
    return make_vectorized_environment(
        config=eval_config.env_config_path,
        route_file=record.route_file,
        run_id=run_id,
        base_dir="outputs/runs",
        vec_normalize=str(results_dir / "final_checkpoint" / "vec_normalize.pkl"),
        training=False,
        norm_reward=False,
        wrap=True,
        seed=seed,
        reward_fn=build_reward_fn(train_config.reward),
    )

# Reload the model specifically for deterministic inference
eval_model = DQN.load(model_save_path)

metrics = evaluate_manifest(
    controller=eval_model,
    env_factory=eval_env_factory,
    manifest=manifest,
    split=eval_config.benchmark.split,
    seeds=eval_config.evaluation_seeds,
    controller_name="DQN_Phase1",
    deterministic=eval_config.benchmark.deterministic,
    episodes_per_scenario=eval_config.benchmark.episodes_per_scenario
)

print(f"Benchmark evaluation completed across {len(metrics)} episodes.")


## 7. Show and Plot Evaluation Metrics


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Convert the evaluation metrics into a pandas DataFrame
df_eval = pd.DataFrame([m.__dict__ for m in metrics])
print("Raw Evaluation Metrics:")
display(df_eval)

# Plot Average Waiting Time and Throughput by Scenario
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_eval.plot.bar(x='scenario_id', y='average_waiting_time', ax=axes[0], title='Average Waiting Time by Scenario', color='skyblue')
axes[0].set_ylabel('Waiting Time (s)')

df_eval.plot.bar(x='scenario_id', y='throughput', ax=axes[1], title='Throughput by Scenario', color='lightgreen')
axes[1].set_ylabel('Vehicles')

plt.tight_layout()
plt.show()


## 8. Compare with Fixed-Time Baseline


In [ ]:
from traffic_drl.baselines.fixed_time import FixedTimeController, make_fixed_time_env
import seaborn as sns

# Initialize Fixed Time baseline
fixed_time_controller = FixedTimeController()

def fixed_time_env_factory(record, seed):
    return make_fixed_time_env(
        record=record,
        config=eval_config.env_config_path,
        run_id=f"{run_id}_fixed_time",
        base_dir="outputs/runs",
    )

print("Evaluating Fixed-Time Baseline...")
metrics_ft = evaluate_manifest(
    controller=fixed_time_controller,
    env_factory=fixed_time_env_factory,
    manifest=manifest,
    split=eval_config.benchmark.split,
    seeds=eval_config.evaluation_seeds,
    controller_name="Fixed-Time",
    deterministic=True,
    episodes_per_scenario=eval_config.benchmark.episodes_per_scenario
)

print(f"Fixed-Time evaluation completed across {len(metrics_ft)} episodes.")

# Combine and compare!
df_all = pd.DataFrame([m.__dict__ for m in metrics + metrics_ft])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(data=df_all, x='scenario_id', y='average_waiting_time', hue='controller', ax=axes[0])
axes[0].set_title('Average Waiting Time Comparison')
axes[0].set_ylabel('Waiting Time (s)')

sns.barplot(data=df_all, x='scenario_id', y='throughput', hue='controller', ax=axes[1])
axes[1].set_title('Throughput Comparison')
axes[1].set_ylabel('Vehicles')

plt.tight_layout()
plt.show()
